ADD EXCEPTIONS AND WHILE LOOP IN ALL PROCESSES THAT INVOLVES USER INPUT
    
data import - ✅
check if the uploaded dataset is a xlxs or csv file - ✅
give a summary of the dataset - ✅
remove any outliers ( winzorisation ) - ✅ 
checking for duplicates and removing - ✅ 
input target seperation - ✅ 
check if the target column is Numeric, binary or categorical - ✅
if regression: - ✅
    data type =  int/ float
    unique values >20
    unique ratio >0.5
    user confirmation
if classification: - ✅
    data type = object, Int
    unique values < 20
    unique ratio < 0.5
    if object : 
        encode them into integer and create a dictinary to store matching values
    user confirmation
if its object and has 20 + unique values - say the target column is invalid - ✅

ask for the ML Type ( regression or classification )
if regression:
    convertion of categorical data into numeric
    use feature importance to remove cols if theres too much
    split into train and test data 
    checking of missing data and filling it
    use all the regressors provided 
    find the best model
    provide the the model through joblib
    ask input if theres a need to predict
    thanks

if classifier:
    check for class imbalance 
    convertion of categorical data into numeric
    use feature importance to remove cols if theres too much
    checking of missing data and filling it
    standardisation ( standard scaler)
    split into train and test data ( stratified k fold ) 
    if class imbalance:
        use Smote on training data alone to stabilize
    use all the classifiers provided 
    find the best model
    provide the the model and scaler through joblib
    ask input if theres a need to predict
    thanks

filepath = "D:\COLLEGE\heart.csv" - classification
filepath = "D:\COLLEGE\Linear Regression - Sheet1.csv" - regression

In [1]:
#IMPORTING THE NECESSARY LIBRARIES..!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [25]:
# DATA IMPORT MODULE..!

def data_ingestion(): 
    data = None
    while data is None:
        try:
            file_path = input ("Give the path to the dataset that you want to be processed (enclose it within quotes):").strip(" \" ")
            
            if file_path.endswith(".csv") is True:
                data = pd.read_csv(file_path)
            elif file_path.endswith(".xlsx") is True:
                data = pd.read_excel(file_path)
            else:
                print("The file needs to be in .csv or .xlsx format")
                continue
                
        except FileNotFoundError:
            print("❌ File not found. Please check the path.")
        except PermissionError:
            print("❌ Permission denied while accessing the file.")
        except AttributeError:
            print("Kindly Enter the correct file path again..!")
        else:
            print(f"\n\nThe Dataset has been successfully uploaded, here's a short preview..!\n\n\n{data.head()}")
    return data
    
data = data_ingestion()

Give the path to the dataset that you want to be processed (enclose it within quotes): "C:\Users\User\Downloads\archive (4)\online_course_engagement_data.csv"




The Dataset has been successfully uploaded, here's a short preview..!


   UserID CourseCategory  TimeSpentOnCourse  NumberOfVideosWatched  \
0    5618         Health          29.979719                     17   
1    4326           Arts          27.802640                      1   
2    5849           Arts          86.820485                     14   
3    4992        Science          35.038427                     17   
4    3866    Programming          92.490647                     16   

   NumberOfQuizzesTaken  QuizScores  CompletionRate  DeviceType  \
0                     3   50.365656       20.860773           1   
1                     5   62.615970       65.632415           1   
2                     2   78.458962       63.812007           1   
3                    10   59.198853       95.433162           0   
4                     0   98.428285       18.102478           0   

   CourseCompletion  
0                 0  
1                 0  
2                 1  
3             

In [26]:
# Gives a description about data..!
def summary_of_data(data):
    columns = data.columns.to_list()
    shape = data.shape
    missing_values = data.isnull().sum()
    duplicates = data.duplicated().sum()

    print(f"The dataset has {shape[0]} rows and {shape[1]} columns.")
    print(f"\nThe column names are : {columns}")
    print(f"\nThe missing values in each attributes are shown below...!\n\n{missing_values}")
    print(F"\n The dataset has {duplicates} duplicate values.")
    print(F" Other things you may want a look into : \n\n{data.describe()}")
summary_of_data(data)

The dataset has 9000 rows and 9 columns.

The column names are : ['UserID', 'CourseCategory', 'TimeSpentOnCourse', 'NumberOfVideosWatched', 'NumberOfQuizzesTaken', 'QuizScores', 'CompletionRate', 'DeviceType', 'CourseCompletion']

The missing values in each attributes are shown below...!

UserID                   0
CourseCategory           0
TimeSpentOnCourse        0
NumberOfVideosWatched    0
NumberOfQuizzesTaken     0
QuizScores               0
CompletionRate           0
DeviceType               0
CourseCompletion         0
dtype: int64

 The dataset has 877 duplicate values.
 Other things you may want a look into : 

            UserID  TimeSpentOnCourse  NumberOfVideosWatched  \
count  9000.000000        9000.000000            9000.000000   
mean   4498.894556          50.163822              10.024667   
std    2596.849433          28.491750               6.029878   
min       1.000000           1.005230               0.000000   
25%    2251.750000          25.440548              

In [27]:
#OUTLIER HANDLING..!

def handling_outliers(df):
    numerical_columns = df.select_dtypes(include=['number']).columns
    for col in numerical_columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
    print(" The Outliers in the dataset have been successfully dealt with by DataForge( if there was any..! )")
        
    return df
data_without_outliers = handling_outliers(data)


 The Outliers in the dataset have been successfully dealt with by DataForge( if there was any..! )


In [28]:
#DUPLICATE REMOVAL..!
def remove_duplicates(df):
    initial_count = len(df)
    df = df.drop_duplicates(keep = "first").reset_index(drop=True)
    final_count = len(df)
    print(f"DataForge Removed {initial_count - final_count} duplicate rows.")
    
    return df

cleaned_data = remove_duplicates(data_without_outliers)

DataForge Removed 877 duplicate rows.


In [29]:
#INPUT TARGET SEPERATION...!
def input_output_separator(df):
    target = None
    while target not in df.columns.to_list():
        target = input ("\nNow for the next part, Correctly type the output column's name :").strip("\"")
        if target not in df.columns.to_list(): 
            print("\nThe input you have given doesnt seem to match any attributes in the dataset, kindly enter it again..!")
        
    input_data = df.columns.to_list()
    input_data.remove(target)
    input_data = df[input_data]
    target = df[target]
    print("DataForge has successfully seperated the dataset into input and target data..!")
    return input_data, target
    
input_data, target = input_output_separator(cleaned_data)


Now for the next part, Correctly type the output column's name : CourseCompletion


DataForge has successfully seperated the dataset into input and target data..!


In [30]:
# TARGET TESTING AND MODEL SUITABILITY MODULE..!
def target_tester(target):
    model = ""
    while model not in ["r", "c"]:
        model = input(
            "Enter the type of ML algorithm you want to perform "
            "(Regression / Classification) : (R / C): ").lower()
        if model not in ["r", "c"]:
            print("Please enter a valid input (R or C):")
            
    target_dtype = target.dtype
    unique_num = target.nunique()
    unique_ratio = unique_num / len(target)

    is_numeric = np.issubdtype(target_dtype, np.number)
    is_object = target_dtype == "object"

    if model == "r":
        if is_numeric and unique_num > 10 and unique_ratio > 0.1:
            print("DataForge will now proceed to process the data for ML processes")
        else:
            confirmation = ""
            while confirmation not in ["y", "n"]:
                confirmation = input(
                    "⚠️ Target may be better suited for Classification. FYI...(T/F) or ('1'/'0') is a classification problem."
                    "Do you want to switch? (Y/N): "
                ).lower()
                if confirmation not in ["y","n"]:
                    print("Enter a valid input either 'Y' or 'N'.")
            if confirmation == "y":
                model = "c"
                print("The model has been changed to classification")
            else:
                print("DataForge will now proceed to process the data for ML processes")

    if model == "c":
        if (is_object) or (is_numeric and unique_num <= 20 and unique_ratio < 0.05):
            print("DataForge will now proceed to process the data for ML processes")
        else:
            confirmation = ""
            while confirmation not in ["y", "n"]:
                confirmation = input(
                    "⚠️ Target may be better suited for Regression. "
                    "Do you want to switch? (Y/N): "
                ).lower()
                if confirmation not in ["y","n"]:
                    print("Enter a valid input either 'Y' or 'N'.")
            if confirmation == "y":
                model = "r"
                print("The model has been changed to regression")
            else:
                print("DataForge will now proceed to process the data for ML processes")

    return model
Model = target_tester(target)

Enter the type of ML algorithm you want to perform (Regression / Classification) : (R / C):  c


DataForge will now proceed to process the data for ML processes


In [ ]:
def reg_model(model, input_val, target_col):
    input_col = pd.get_dummies(input_val,dtype = int)
    x_train, X_test, y_train, Y_test = train_test_split(input_col, target_col,test_size = 0.2, random_state =42)
    for col in x_train.columns:
        x_train[col] = x_train[col].fillna(x_train[col].mean())
    y_train = y_train.fillna(y_train.mean())
    L_reg = LinearRegression()
    R_reg = RandomForestRegressor(n_estimators=100, random_state=42)
    G_reg = GradientBoostingRegressor(random_state=42)
    L_reg.fit(x_train,y_train)
    R_reg.fit(x_train,y_train)
    G_reg.fit(x_train,y_train)
    L_pred = L_reg.predict(X_test)
    R_pred = R_reg.predict(X_test)
    G_pred = G_reg.predict(X_test)
    score_L = round(r2_score(Y_test, L_pred),3)
    score_R = round(r2_score(Y_test, R_pred),3)
    score_G = round(r2_score(Y_test, G_pred),3)
    print(f"Linear regression's accuracy : {score_L}\nRandom forest regressor's accuracy: {score_R}\nGradient boost regressor's accuracy: {score_G}.")
reg_model( Model, input_data, target)

if classifier:
    check for class imbalance 
    convertion of categorical data into numeric
    use feature importance to remove cols if theres too much
    checking of missing data and filling it
    standardisation ( standard scaler)
    split into train and test data ( stratified k fold ) 
    if class imbalance:
        use Smote on training data alone to stabilize
    use all the classifiers provided 
    find the best model
    provide the the model and scaler through joblib
    ask input if theres a need to predict
    thanks

In [31]:
def Cls_model(model, input_val, target_col):
    #class imbalance module
    
    cls_bal = target_col.value_counts().to_dict()
    final_df = pd.get_dummies(input_val, dtype = int)
    print("\nAfter the convertion of categorical data into numeric, the dataset looks like :\n", final_df.head())
    #Splitting
    
    X_train,X_test,Y_train,Y_test = train_test_split(final_df, target_col, test_size = 0.2, random_state = 42)
    #filling NA
    
    for col in X_train.columns:
        X_train[col] = X_train[col].fillna(X_train[col].mean())
        X_test[col] = X_test[col].fillna(X_train[col].mean())
    Y_train = Y_train.fillna(Y_train.mean())
    Y_test = Y_test.fillna(Y_train.mean())
    #Oversampling
    
    populate = ""
    while populate not in [ "y","n" ]:
        populate = input(f"\nDo you want to use SMOTE to populate the dataset..?\n The class balance is {cls_bal} : (Y/N)").lower()
        if populate == "y":
            smote = SMOTE(sampling_strategy='minority', random_state=42)
            X_train, Y_train = smote.fit_resample(X_train, Y_train)
            print("Dataforge has successfully applied oversampling using smote..")
        elif populate == "n":
            print("Let's move on then..")
        else:
            print("Enter a proper input, either 'Y' or 'N'.")
    print("Dataforge next moves on to Scaling..")
    #scaling
    scale = StandardScaler()
    X_train_scaled = scale.fit_transform(X_train)
    X_test_scaled = scale.transform(X_test)
    print("Dataforge has successfully scaled the data using standard scaler...")
    return X_train_scaled, X_test_scaled, Y_train, Y_test
    
X_train, X_test, Y_train, Y_test = Cls_model(Model, input_data, target )
    


After the convertion of categorical data into numeric, the dataset looks like :
    UserID  TimeSpentOnCourse  NumberOfVideosWatched  NumberOfQuizzesTaken  \
0    5618          29.979719                     17                     3   
1    4326          27.802640                      1                     5   
2    5849          86.820485                     14                     2   
3    4992          35.038427                     17                    10   
4    3866          92.490647                     16                     0   

   QuizScores  CompletionRate  DeviceType  CourseCategory_Arts  \
0   50.365656       20.860773           1                    0   
1   62.615970       65.632415           1                    1   
2   78.458962       63.812007           1                    1   
3   59.198853       95.433162           0                    0   
4   98.428285       18.102478           0                    0   

   CourseCategory_Business  CourseCategory_Health  CourseC


Do you want to use SMOTE to populate the dataset..?
 The class balance is {0: 4555, 1: 3568} : (Y/N) y


Dataforge has successfully applied oversampling using smote..
Dataforge next moves on to Scaling..
Dataforge has successfully scaled the data using standard scaler...


In [32]:
def cls_model_train(X_train, X_test, Y_train, Y_test):
    
    log_model = LogisticRegression(max_iter=1000, random_state = 42, class_weight= "balanced")
    log_model.fit(X_train, Y_train)
    log_pred = log_model.predict(X_test)
    
    rf_model = RandomForestClassifier(n_estimators=500, random_state=42, class_weight= "balanced")
    rf_model.fit(X_train, Y_train)
    rf_pred = rf_model.predict(X_test)
    
    knn = KNeighborsClassifier(n_neighbors=8)
    knn.fit(X_train, Y_train)
    knn_pred = knn.predict(X_test)
    
    print("Dataforge has trained three models on the given datasets and here are the results..\n")
    print("The Metrics for logistic regression are given below:\n",classification_report(log_pred, Y_test))
    print("The Metrics for random forest classifier are given below:\n",classification_report(rf_pred, Y_test))
    print("The Metrics for K- Nearest Neighbours are given below:\n",classification_report(knn_pred, Y_test))
    
cls_model_train(X_train, X_test, Y_train, Y_test)

Dataforge has trained three models on the given datasets and here are the results..

The Metrics for logistic regression are given below:
               precision    recall  f1-score   support

           0       0.82      0.80      0.81       912
           1       0.75      0.77      0.76       713

    accuracy                           0.79      1625
   macro avg       0.79      0.79      0.79      1625
weighted avg       0.79      0.79      0.79      1625

The Metrics for random forest classifier are given below:
               precision    recall  f1-score   support

           0       0.96      0.97      0.97       882
           1       0.97      0.95      0.96       743

    accuracy                           0.96      1625
   macro avg       0.96      0.96      0.96      1625
weighted avg       0.96      0.96      0.96      1625

The Metrics for K- Nearest Neighbours are given below:
               precision    recall  f1-score   support

           0       0.88      0.81    